In [ ]:
# ===============================================
# CELL 1: SETUP & INSTALL PACKAGES — FULL FIX
# ===============================================

print("📦 Đang gỡ các thư viện xung đột (Pandas/Numpy 2.x)...")
!pip uninstall -y hdbscan sentence-transformers huggingface_hub gensim scipy bertopic umap-learn pandas scikit-learn numpy -q

print("📦 Đang cài đặt bộ thư viện tương thích (Numpy 1.x)...")
# Cài đúng thứ tự, pin version tương thích nhau tuyệt đối
!pip install -q numpy==1.26.4
!pip install -q pandas==2.1.4
!pip install -q scikit-learn==1.3.2
!pip install -q scipy==1.11.4
!pip install -q gensim==4.3.2
!pip install -q huggingface_hub==0.21.0
!pip install -q transformers==4.36.0
!pip install -q sentence-transformers==2.7.0
!pip install -q umap-learn==0.5.5
!pip install -q hdbscan==0.8.33 --no-binary hdbscan
!pip install -q bertopic==0.16.0
!pip install -q tqdm

print("\n✅ CÀI ĐẶT THÀNH CÔNG!")
print("⚠️ BẮT BUỘC: Hãy lên menu Kaggle -> Chọn Run -> Restart Session (hoặc nút Restart bên cạnh RAM/CPU) trước khi chạy Cell 2.")

import torch
print("\n" + "="*50)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*50)

📦 Đang gỡ các thư viện xung đột (Pandas/Numpy 2.x)...
📦 Đang cài đặt bộ thư viện tương thích (Numpy 1.x)...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.36.0 requires huggingface-hub<1.0,>=0.19.3, which is not installed.
torchtune 0.6.1 requires huggingface_hub[hf_transfer], which is not installed.
configspace 1.2.2 requires scipy, which is not installed.
woodwork 0.31.0 requires pandas>=2.0.0, which is not installed.
woodwork 0.31.0 requires scikit-learn>=1.1.0, which is not installed.
woodwork 0.31.0 requires scipy>=1.10.0, which is not installed.
boruta 0.4.3 requires scikit-learn>=0.17.1, which is not installed.
boruta 0.4.3 requires scipy>=0.17.0, which is not installed.
nilearn 0.13.1 requires pandas>=2.2.0, which is not installed.
nilearn 0.13.1 requires scikit-learn>=1.4.0, which is not installed.
nilearn 0.13.1 requires scipy>=1.9.0, 

In [ ]:
import pandas as pd
import os
import sys

# FIX: path đến FOLDER chứa file, không phải path đến file
sys.path.append("/kaggle/input/datasets/hoangquancs04222/data-proccessed-score")
from bertopic_model import VietnameseBERTopicModel

print("📊 LOADING DATA")
print("="*50)

DATA_PATH = "/kaggle/input/datasets/hoangquancs04222/data-proccessed-score/stg_posts_core.csv"

COL_NAMES = [
    "post_id", "source", "type", "author", "parent_id",
    "title", "body", "segmented_text",
    "col9", "col10", "col11", "col12",
    "created_at", "crawled_at", "col15", "url"
]

df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=COL_NAMES,
    sep=",",
    on_bad_lines="skip",
    engine="python"
)

print(f"✅ Loaded: {DATA_PATH}")
print(f"   Shape: {df.shape}")

documents = (
    df["segmented_text"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

post_ids = df.loc[df["segmented_text"].notna(), "post_id"].astype(str).tolist()

valid_mask = [len(d) > 10 for d in documents]
documents = [d for d, ok in zip(documents, valid_mask) if ok]
post_ids  = [p for p, ok in zip(post_ids, valid_mask) if ok]

print(f"\n📈 DATA STATS:")
print(f"   Total documents: {len(documents):,}")
print(f"   Avg length: {sum(len(d) for d in documents)/len(documents):.0f} chars")
print(f"\n📝 SAMPLE:")
for i, doc in enumerate(documents[:3]):
    print(f"   {i+1}. {doc[:120]}...")

In [ ]:
# ===============================================
# CELL 3: COMPUTE PHOBERT EMBEDDINGS (CACHE) - FIXED BUG
# ===============================================

import numpy as np, random, time, torch
from sklearn.preprocessing import normalize

print("🧪 COMPUTING PHOBERT EMBEDDINGS")
print("="*50)

# Cố định seed để các lần chạy tuning đều công bằng
random.seed(42) 
TUNING_SAMPLE_SIZE = min(5000, len(documents))

# FIX BUG NGHIÊM TRỌNG: Lấy mẫu qua list index để đảm bảo Map ID - Content chính xác 100%
indices = random.sample(range(len(documents)), TUNING_SAMPLE_SIZE)
tuning_docs = [documents[i] for i in indices]
tuning_post_ids = [post_ids[i] for i in indices]

print(f"\n✅ Đã lấy mẫu {len(tuning_docs):,} dòng ngẫu nhiên và KHỚP ID tuyệt đối!")

# Dùng VietnameseBERTopicModel.encode() — nhất quán với production
tmp_model = VietnameseBERTopicModel(verbose=True)

start = time.time()
embeddings = tmp_model.encode(tuning_docs, batch_size=32)

# Chuẩn hóa L2 (tối ưu cho cosine similarity của UMAP)
embeddings = normalize(embeddings)

print(f"\n✅ Embeddings: {embeddings.shape}, dtype: {embeddings.dtype}")
print(f"   Time: {(time.time()-start)/60:.1f} min")
print("✅ Embeddings đã được chuẩn hóa L2!")

# Giải phóng VRAM tránh OOM cho các bước sau
del tmp_model
torch.cuda.empty_cache()

In [ ]:
# ===============================================
# CELL 4-5-6: EXPERIMENTS (giữ nguyên từ notebook)
# ===============================================

# Chỉ cần thêm import ở đầu nếu chưa có
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# ---- Paste nguyên Cell 4 (experiments list) từ notebook vào đây ----
# ===============================================
# CELL 4: DEFINE EXPERIMENT GRID (UMAP FOCUSED)
# ===============================================

print("🔬 EXPERIMENT GRID - UMAP OPTIMIZATION")
print("="*50)

# Cố định HDBSCAN để đo lường độ hiệu quả của việc giảm chiều (UMAP)
experiments = [
    # 1. Base (Mốc chuẩn - Nén xuống 5 chiều)
    {
        'name': 'umap_base_5d',
        'n_neighbors': 30,
        'n_components': 5,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'Standard compression (5D)'
    },
    # 2. Base (Nén xuống 10 chiều - giống code cũ của bạn)
    {
        'name': 'umap_base_10d',
        'n_neighbors': 30,
        'n_components': 10,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'Lower compression (10D) - Harder for HDBSCAN'
    },
    # 3. Nén cực đại (3 chiều - Giống không gian 3D, siêu đặc)
    {
        'name': 'umap_max_compress_3d',
        'n_neighbors': 30,
        'n_components': 3,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'Extreme compression (3D) - Denser clusters'
    },
    # 4. Global View (Nhìn bao quát, gom các cụm nhỏ)
    {
        'name': 'umap_global_view',
        'n_neighbors': 100, 
        'n_components': 5,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'High n_neighbors (100) for global structure'
    },
    # 5. Local View (Tập trung vi mô, dễ xé nhỏ topic)
    {
        'name': 'umap_local_view',
        'n_neighbors': 15,
        'n_components': 5,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'Low n_neighbors (15) for high detail'
    },
    # 6. Bộ cấu hình thực dụng cho Dashboard (Gom 75 Topics + UMAP 5D)
    {
        'name': 'umap_optimized_force_75',
        'n_neighbors': 40,
        'n_components': 5,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 75,
        'description': 'The ultimate dashboard setup'
    }
]

# Print summary
print(f"Total experiments: {len(experiments)}")
print("\n" + "-"*95)
print(f"{'Name':<25} {'n_nbr':>6} {'n_comp':>6} {'m_dist':>6} {'m_clust':>7} {'m_samp':>6} {'nr_top':>8}")
print("-" * 95)
for exp in experiments:
    print(f"{exp['name']:<25} {exp['n_neighbors']:>6} {exp['n_components']:>6} {exp['min_dist']:>6} {exp['min_cluster_size']:>7} {exp['min_samples']:>6} {str(exp['nr_topics']):>8}")
print("-" * 95)
# ---- Paste nguyên Cell 5 (run experiments loop) vào đây ----
# ===============================================
# CELL 5: RUN EXPERIMENTS
# ===============================================

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import time
import warnings
warnings.filterwarnings('ignore')

print('\n🏋️ RUNNING EXPERIMENTS')
print('='*50)
print(f'Documents: {len(tuning_docs):,}')
print(f'Embeddings shape: {embeddings.shape}')
print(f'Experiments: {len(experiments)}')
print('='*50)

results = []

def calculate_diversity(topic_model, top_n=10):
    try:
        topics = topic_model.get_topics()
        all_words = []
        for topic_id, words in topics.items():
            if topic_id != -1:
                all_words.extend([w[0] for w in words[:top_n]])
        if not all_words:
            return 0.0
        return len(set(all_words)) / len(all_words)
    except Exception:
        return 0.0

def calculate_umass(topic_model, texts):
    try:
        topics_dict = {tid: words for tid, words in topic_model.get_topics().items() if tid != -1}
        if not topics_dict:
            return 0.0
        topics_words = [[w for w, _ in words[:10]] for words in topics_dict.values()]
        tokenized = [[t for t in doc.split() if len(t) > 1] for doc in texts]
        tokenized = [t for t in tokenized if t]
        dictionary = Dictionary(tokenized)
        dictionary.filter_extremes(no_below=2, no_above=0.95)
        cm = CoherenceModel(
            topics=topics_words, texts=tokenized, dictionary=dictionary, coherence='u_mass'
        )
        return cm.get_coherence()
    except Exception:
        return 0.0

for i, exp in enumerate(experiments):
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(experiments)}] {exp['name']}")
    print(f"    {exp['description']}")
    
    start_time = time.time()

    try:
        # CẬP NHẬT: Nhận diện linh hoạt n_components từ dictionary
        umap_model = UMAP(
            n_neighbors=exp['n_neighbors'],
            n_components=exp.get('n_components', 5), # Mặc định là 5 nếu quên cấu hình
            min_dist=exp['min_dist'],
            metric='cosine',
            random_state=42,
        )
        
        hdbscan_model = HDBSCAN(
            min_cluster_size=exp['min_cluster_size'],
            min_samples=exp['min_samples'],
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
        )

        nr_topics_val = None if exp['nr_topics'] == 'auto' else int(exp['nr_topics'])

        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            nr_topics=nr_topics_val,
            calculate_probabilities=False,
            verbose=False,
        )

        topics, probs = topic_model.fit_transform(tuning_docs, embeddings=embeddings)

        training_time = time.time() - start_time

        topic_info = topic_model.get_topic_info()
        n_topics = len([t for t in topic_info['Topic'] if t != -1])
        n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].sum() if -1 in topic_info['Topic'].values else 0
        
        outlier_ratio = n_outliers / len(tuning_docs)
        diversity = calculate_diversity(topic_model)
        coherence_umass = calculate_umass(topic_model, tuning_docs)

        result = {
            'name': exp['name'],
            'description': exp['description'],
            'n_neighbors': exp['n_neighbors'],
            'n_components': exp.get('n_components', 5),
            'min_dist': exp['min_dist'],
            'min_cluster_size': exp['min_cluster_size'],
            'min_samples': exp['min_samples'],
            'nr_topics': str(exp['nr_topics']),
            'n_topics_found': n_topics,
            'n_outliers': n_outliers,
            'outlier_ratio': outlier_ratio,
            'diversity': diversity,
            'coherence_umass': coherence_umass,
            'training_time': training_time,
        }
        results.append(result)

        print(f'    ✅ {training_time:.1f}s | Topics: {n_topics} | '
              f'Outliers: {outlier_ratio*100:.1f}% | '
              f'Diversity: {diversity:.3f} | '
              f'U_Mass: {coherence_umass:.4f}')

        pd.DataFrame(results).to_csv('/kaggle/working/tuning_progress.csv', index=False)

    except Exception as e:
        print(f'    ❌ FAILED: {e}')
        results.append({
            'name': exp['name'],
            'error': str(e),
        })

print('\n' + '='*50)
print(f'✅ COMPLETED {len(results)} EXPERIMENTS')
print('='*50)

# ---- Paste nguyên Cell 6 (analyze results) vào đây ----
# ===============================================
# CELL 6: ANALYZE RESULTS (NEW BUSINESS LOGIC)
# ===============================================

import pandas as pd

print('\n📊 EXPERIMENT RESULTS ANALYSIS (DASHBOARD-ORIENTED)')
print('='*60)

results_df = pd.DataFrame(results)
valid_results = results_df[results_df['n_topics_found'].notna()].copy()

def composite_score(row):
    """
    Hàm tính điểm mới: Ưu tiên tạo ra số lượng topic đủ lớn (50-100) cho Dashboard,
    chấp nhận tỷ lệ nhiễu (outliers) ở mức 30-45% để lọc rác MXH.
    """
    # 1. Topic Count Score (Trọng số cao nhất: 35%)
    n = row['n_topics_found']
    if 50 <= n <= 100:
        topic_score = 1.0  # Vùng lý tưởng nhất
    elif n < 50:
        topic_score = max(0.0, n / 50.0)  # Phạt nặng nếu quá ít topic (vì không có ích cho phân tích)
    else:
        topic_score = max(0.0, 1.0 - (n - 100) / 200.0) # Phạt nhẹ nếu quá nhiều (trên 100 có thể hơi loãng)
    topic_score *= 0.35

    # 2. Outlier Score (Trọng số: 20%)
    outlier_ratio = row['outlier_ratio']
    if 0.30 <= outlier_ratio <= 0.45:
        outlier_score = 1.0  # Mức lọc rác hoàn hảo cho Social Media
    elif outlier_ratio < 0.30:
        outlier_score = max(0.0, outlier_ratio / 0.30)  # Cố ép rác vào cụm -> Trừ điểm
    else:
        outlier_score = max(0.0, 1.0 - (outlier_ratio - 0.45) / 0.55) # Outlier trên 45% -> Mất mát dữ liệu -> Trừ điểm
    outlier_score *= 0.20

    # 3. Diversity Score (Trọng số: 25%) - Càng đa dạng từ vựng giữa các topic càng tốt
    diversity_score = row['diversity'] * 0.25

    # 4. Coherence (U_Mass) Score (Trọng số: 20%) - Đưa về thang [0, 1] từ khoảng [-20, 0]
    u_mass_clipped = max(-20.0, min(0.0, row.get('coherence_umass', -20.0)))
    coherence_score = ((u_mass_clipped + 20.0) / 20.0) * 0.20

    return topic_score + outlier_score + diversity_score + coherence_score


valid_results['composite_score'] = valid_results.apply(composite_score, axis=1)
valid_results = valid_results.sort_values('composite_score', ascending=False)

print('\n📋 ALL RESULTS (sorted by NEW composite score):')
print('-'*115)
display_cols = ['name', 'n_topics_found', 'outlier_ratio', 'diversity', 'coherence_umass', 'composite_score']
print(valid_results[display_cols].to_string(index=False))
print('-'*115)

print('\n🏆 TOP 3 CONFIGURATIONS FOR DASHBOARD:')
for i, (_, row) in enumerate(valid_results.head(3).iterrows()):
    print(f"\n#{i+1}: {row['name']}")
    print(f"    Topics: {row['n_topics_found']} | "
          f"Outliers: {row['outlier_ratio']*100:.1f}% | "
          f"Diversity: {row['diversity']:.3f} | "
          f"U_Mass: {row['coherence_umass']:.4f}")
    print(f"    n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, "
          f"min_cluster_size={row['min_cluster_size']}, min_samples={row['min_samples']}")
    print(f"    Composite Score: {row['composite_score']:.4f}")

best_config = valid_results.iloc[0]
print('\n' + '='*60)
print(f"🥇 BEST CONFIG TO USE: {best_config['name']}")
print('='*60)

In [ ]:
# ===============================================
# CELL 7: TRAIN FINAL MODEL — với post_ids thực
# ===============================================

# --- Paste nguyên Cell 7 từ notebook ---
# ===============================================
# CELL 7: TRAIN FINAL MODEL WITH BEST CONFIG
# ===============================================

print("🏆 TRAINING FINAL MODEL")
print("="*50)
print(f"Using config: {best_config['name']}")
print(f"Documents: {len(tuning_docs):,}")

# Extract best hyperparameters
best_params = {
    'n_neighbors': int(best_config['n_neighbors']),
    'min_dist': float(best_config['min_dist']),
    'min_cluster_size': int(best_config['min_cluster_size']),
    'min_samples': int(best_config['min_samples']),
    'nr_topics': best_config['nr_topics']
}

print(f"\nHyperparameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# Create final model
print("\n[1/4] Creating UMAP model...")
umap_model = UMAP(
    n_neighbors=best_params['n_neighbors'],
    n_components=5,
    min_dist=best_params['min_dist'],
    metric='cosine',
    random_state=42
)

print("[2/4] Creating HDBSCAN model...")
hdbscan_model = HDBSCAN(
    min_cluster_size=best_params['min_cluster_size'],
    min_samples=best_params['min_samples'],
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

print("[3/4] Creating BERTopic model...")
nr_topics = None if best_params['nr_topics'] == 'auto' else int(best_params['nr_topics'])
final_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=nr_topics,
    calculate_probabilities=True,
    verbose=True
)

print("[4/4] Fitting model...")
start_time = time.time()
topics, probs = final_model.fit_transform(tuning_docs, embeddings=embeddings)
training_time = time.time() - start_time

# Results
topic_info = final_model.get_topic_info()
n_topics = len([t for t in topic_info['Topic'] if t != -1])
n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].sum() if -1 in topic_info['Topic'].values else 0

print("\n" + "="*50)
print("✅ FINAL MODEL TRAINED!")
print("="*50)
print(f"Topics found: {n_topics}")
print(f"Outliers: {n_outliers} ({n_outliers/len(tuning_docs)*100:.1f}%)")
print(f"Training time: {training_time:.1f} seconds")

# Show topics
print("\n📋 TOPIC SUMMARY:")
print(topic_info[['Topic', 'Count', 'Name']].head(15).to_string(index=False))
# Sau đó THÊM phần export bên dưới:

# Tạo final_model dùng VietnameseBERTopicModel để dùng export methods
final_vm = VietnameseBERTopicModel(
    n_neighbors=int(best_config['n_neighbors']),
    n_components=5,
    min_dist=float(best_config['min_dist']),
    min_cluster_size=int(best_config['min_cluster_size']),
    min_samples=int(best_config['min_samples']),
    verbose=True
)
# Truyền embeddings đã cache → không encode lại
final_vm.topic_model = final_model   # reuse model đã train ở Cell 7
final_vm.topics_ = topics
final_vm.probs_  = probs

print("\n✅ VietnameseBERTopicModel wrapper ready for export")